# T1 Bench Validation

Load bench CSV, compare to simulation, run T1 ±30% gate.

Protocol: `docs/SGH1_TEST_PROTOCOL.md`

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from simulation.bench_validation import validate_bench_csv, export_validation
from simulation.constants import C_BRINE_8PCT, C_TREATED_WW
from simulation.pro_cycle import steady_state_pro

In [ ]:
# Predicted baseline from experiments export
exp_path = ROOT / 'exports' / 'paper_experiments.json'
if exp_path.exists():
    exp = json.loads(exp_path.read_text())
    baseline = exp['sg_h1_baseline']
    print('Sim baseline P (default L_p):', baseline['P_default_Lp_W'], 'W')
    print('Δπ (MPa):', baseline['delta_pi_MPa'])
else:
    st = steady_state_pro(C_BRINE_8PCT, C_TREATED_WW, 0.72)
    baseline = {'P_default_Lp_W': st.P_elec_equiv_W, 'delta_pi_MPa': st.delta_pi / 1e6}

In [ ]:
# CSV path — use latest bench run or dry-run sim CSV
DATA = ROOT / 'data' / 'bench'
csv_files = sorted(DATA.glob('*.csv'))
CSV_PATH = csv_files[-1] if csv_files else None
if CSV_PATH is None:
    raise FileNotFoundError('No CSV in data/bench — run: python -m daq.logger --test T1 --duration 120')
print('Using', CSV_PATH)

In [ ]:
result = validate_bench_csv(CSV_PATH)
out = export_validation(CSV_PATH)
status = 'PASS' if result.pass_t1 else 'FAIL'
print(f'\n=== T1 VALIDATION: {status} ===')
print(f"P'' measured: {result.P_density_measured_W_m2:.2f} W/m²")
print(f"P'' predicted: {result.P_density_predicted_W_m2:.2f} W/m²")
print(f'Relative error: {result.relative_error_pct:.1f}%')
print(f'P_net steady: {result.P_net_steady_W:.3f} W')
print(f'L_p fit: {result.L_p_fit_m_Pa_s:.2e} m/(Pa·s)')
print(f'Exported → {out}')

In [ ]:
df = pd.read_csv(CSV_PATH)
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(df['t_s'], df['P_elec_W'], label='P_elec')
if 'P_net_W' in df.columns:
    axes[0].plot(df['t_s'], df['P_net_W'], label='P_net', alpha=0.8)
axes[0].set_ylabel('Power (W)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(df['t_s'], df['cond_feed_mS_cm'], label='feed')
axes[1].plot(df['t_s'], df['cond_draw_mS_cm'], label='draw')
axes[1].set_ylabel('Conductivity (mS/cm)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[2].plot(df['t_s'], df['P_draw_bar'] - df['P_feed_bar'], label='ΔP (bar)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('ΔP (bar)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
fig.suptitle(f'T1 bench run — {CSV_PATH.name}')
plt.tight_layout()
plt.show()